# 13.5. 대칭 적응 선형 결합

In [1]:
import numpy as np


def C(n, axis="z"):
    th = 2 * np.pi / n
    c, s = np.cos(th), np.sin(th)
    if axis == "z":
        return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1.0]])
    if axis == "y":
        return np.array([[c, 0, s], [0, 1.0, 0], [-s, 0, c]])
    return np.array([[1.0, 0, 0], [0, c, -s], [0, s, c]])


def sigma(plane):
    m = np.eye(3)
    m[{"yz": 0, "xz": 1, "xy": 2}[plane]] *= -1
    return m


def sigma_v(d):
    th = np.radians(d)
    return np.array([[np.cos(2 * th), np.sin(2 * th), 0], [np.sin(2 * th), -np.cos(2 * th), 0], [0, 0, 1.0]])


def orbital_rep(op, coords, labels):
    n = len(coords)
    M = np.zeros((n, n))
    moved = coords @ op.T
    for i, (p, l) in enumerate(zip(moved, labels)):
        for j, (q, m) in enumerate(zip(coords, labels)):
            if l == m and np.allclose(p, q, atol=1e-6):
                M[j, i] = 1
                break
    return M


def project(reps, chars, vec):
    """사영 연산자: sum(chi_i(R) * R) * d_i / h"""
    P = sum(c * D for c, D in zip(chars, reps)) * chars[0] / len(reps)
    return P @ vec


def combo(v, names):
    """계수 벡터를 보기 좋게 쓴다"""
    v = np.where(np.abs(v) < 1e-10, 0, v)
    if not v.any():
        return "0"
    terms = []
    for c, n in zip(np.round(v, 3), names):
        if c == 0:
            continue
        sign = "+" if c > 0 else "-"
        coef = "" if abs(c) == 1 else f"{abs(c):g}"
        terms.append(f"{sign} {coef}{n}")
    return " ".join(terms).removeprefix("+ ")


r, theta = 0.9572, np.radians(104.52)
water = np.array(
    [
        [0, 0, 0],
        [0, r * np.sin(theta / 2), -r * np.cos(theta / 2)],
        [0, -r * np.sin(theta / 2), -r * np.cos(theta / 2)],
    ]
)
water_names = ["s_O", "s_H1", "s_H2"]
water_reps = [orbital_rep(op, water, ["O", "H", "H"]) for op in [np.eye(3), C(2), sigma("xz"), sigma("yz")]]

c2v = {"A1": [1, 1, 1, 1], "A2": [1, 1, -1, -1], "B1": [1, -1, 1, -1], "B2": [1, -1, -1, 1]}

phi_1 = np.array([0, 1, 0])
for name, chars in c2v.items():
    print(f"{name}: {combo(project(water_reps, chars, phi_1), water_names)}")

A1: 0.5s_H1 + 0.5s_H2
A2: 0
B1: 0
B2: 0.5s_H1 - 0.5s_H2


In [2]:
r2, z = 0.9375, -0.3810
ammonia = np.array(
    [
        [0, 0, 0],
        [r2, 0, z],
        [r2 * np.cos(2 * np.pi / 3), r2 * np.sin(2 * np.pi / 3), z],
        [r2 * np.cos(4 * np.pi / 3), r2 * np.sin(4 * np.pi / 3), z],
    ]
)

nh3_names = ["s_N", "s_A", "s_B", "s_C"]
nh3_reps = [
    orbital_rep(op, ammonia, ["N", "H", "H", "H"])
    for op in [np.eye(3), C(3), C(3) @ C(3), sigma_v(0), sigma_v(120), sigma_v(240)]
]

c3v = {"A1": [1, 1, 1, 1, 1, 1], "A2": [1, 1, 1, -1, -1, -1], "E": [2, -1, -1, 0, 0, 0]}

s_A = np.array([0, 1, 0, 0])
for name, chars in c3v.items():
    print(f"{name}: {combo(project(nh3_reps, chars, s_A), nh3_names)}")

A1: 0.333s_A + 0.333s_B + 0.333s_C
A2: 0
E: 0.667s_A - 0.333s_B - 0.333s_C


In [3]:
first_basis = project(nh3_reps, chars, s_A)
second_basis = project(nh3_reps, c3v["E"], np.array([0, 0, 1, 0]))
print(combo(second_basis, nh3_names))
print(round(first_basis @ second_basis, 4))

- 0.333s_A + 0.667s_B - 0.333s_C
-0.3333


In [4]:
second_basis = second_basis - (second_basis @ first_basis) / (first_basis @ first_basis) * first_basis
print(combo(second_basis, nh3_names))

0.5s_B - 0.5s_C


In [5]:
R = 1.39
benzene = np.array([[R * np.cos(np.pi / 3 * k), R * np.sin(np.pi / 3 * k), 0] for k in range(6)])
benzene_names = [f"p{k + 1}" for k in range(6)]


def Rz(theta):
    """z축 회전 행렬"""
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1.0]])


benzene_ops, benzene_classes = [], []
for k, cls in enumerate(["E", "C6", "C3", "C2", "C3", "C6"]):
    benzene_ops.append(Rz(np.pi / 3 * k))
    benzene_classes.append(cls)
for d in [0, 60, 120]:
    benzene_ops.append(sigma_v(d))
    benzene_classes.append("σv")
for d in [30, 90, 150]:
    benzene_ops.append(sigma_v(d))
    benzene_classes.append("σd")

benzene_reps = [orbital_rep(op, benzene, ["C"] * 6) for op in benzene_ops]

classes = ["E", "C6", "C3", "C2", "σv", "σd"]
sizes = [1, 2, 2, 1, 3, 3]
c6v = {
    "A1": [1, 1, 1, 1, 1, 1],
    "A2": [1, 1, 1, 1, -1, -1],
    "B1": [1, -1, 1, -1, 1, -1],
    "B2": [1, -1, 1, -1, -1, 1],
    "E1": [2, 1, -1, -2, 0, 0],
    "E2": [2, -1, -1, 2, 0, 0],
}


def reduce(chi, table, sizes):
    return {name: round(sum(g * a * b for g, a, b in zip(sizes, chi, row)) / sum(sizes)) for name, row in table.items()}


chi = [round(np.trace(benzene_reps[benzene_classes.index(c)])) for c in classes]
print("유형별 지표:", chi)
print("축약 결과:", reduce(chi, c6v, sizes))

유형별 지표: [6, 0, 0, 0, 2, 0]
축약 결과: {'A1': 1, 'A2': 0, 'B1': 1, 'B2': 0, 'E1': 1, 'E2': 1}


In [6]:
def op_chars(name):
    row = dict(zip(classes, c6v[name]))
    return [row[c] for c in benzene_classes]


p1, p2 = np.array([1, 0, 0, 0, 0, 0]), np.array([0, 1, 0, 0, 0, 0])
salc = {}
for name in c6v:
    v = project(benzene_reps, op_chars(name), p1)
    print(f"{name:>3}: {combo(v, benzene_names)}")
    if v.any():
        salc[name] = v

print()
for name in ["E1", "E2"]:
    w = project(benzene_reps, op_chars(name), p2)
    w = w - (w @ salc[name]) / (salc[name] @ salc[name]) * salc[name]
    salc[name + "'"] = w
    print(f"{name}의 짝: {combo(w, benzene_names)}")

 A1: 0.167p1 + 0.167p2 + 0.167p3 + 0.167p4 + 0.167p5 + 0.167p6
 A2: 0
 B1: 0.167p1 - 0.167p2 + 0.167p3 - 0.167p4 + 0.167p5 - 0.167p6
 B2: 0
 E1: 0.333p1 + 0.167p2 - 0.167p3 - 0.333p4 - 0.167p5 + 0.167p6
 E2: 0.333p1 - 0.167p2 - 0.167p3 + 0.333p4 - 0.167p5 - 0.167p6

E1의 짝: 0.25p2 + 0.25p3 - 0.25p5 - 0.25p6
E2의 짝: 0.25p2 - 0.25p3 + 0.25p5 - 0.25p6
